# Qdrant × Future AGI: Fixing RAG Decay

The app is intentionally broken against a live Qdrant collection. Each section diagnoses one retrieval failure, applies a Qdrant fix, and checks that the metric moves.

- **App** (`app.py`): show the failure and the recovered answer
- **Notebook**: apply the fixes
- **Future AGI**: trace, judge, and compare runs

The notebook prints fast gold-label retrieval checks. Judged answer quality lives in Future AGI.

## 0 · Setup

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

import agent
from helpers import config, embeddings
from helpers.embeddings import dense, sparse, colbert

load_dotenv()
client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
COLLECTION = config.COLLECTION

# Start every run from the broken baseline: small dense model, no filter.
agent.set_retrieval(mode="minilm", current_only=False)
embeddings.warmup()   # load all models now, never during a live question
client.count(COLLECTION)

### Future AGI Tracing

`agent.py` registers tracing once:

```python
trace_provider = register(project_type=ProjectType.OBSERVE, project_name="pokedex-rag")
LangChainInstrumentor().instrument(tracer_provider=trace_provider)
```

Any process that imports the agent exports traces to `pokedex-rag`: this notebook, the Streamlit app, and rehearsal scripts. Qdrant calls appear as retriever spans.

## 1 · The Pain: Retrieval Decayed As The Corpus Grew

The same golden queries are scored at three corpus sizes: Gen 1 only, Gen 1-4, and the full dex. `scaling_curve.py` generated the chart from the documents that existed at each stage.

Recall@5 falls 0.67 → 0.39 as lookalike species accumulate. Duplicate rate is high from day one; growth makes the latent flaws visible.

The curve uses all scored queries. Later sections use slices of the same golden set, so their baselines differ.

In [ ]:
from IPython.display import Image
Image("data/scaling_curve.png")   # generated offline by scaling_curve.py

## 2 · Fix #1: Dedup The Collection

**Dashboard read:** groundedness and context relevance can still look fine, but the retrieval panel shows the same `doc_id` filling the top-k. That is duplication, not a model problem.

In [ ]:
# Duplicate rate in the top-k before the fix: a slot is wasted if it repeats a fragment
# with the same doc_id and text we already retrieved.
def dup_rate(query):
    chunks = agent.retrieve(query, mode="minilm")
    keys = [(c["doc_id"], c["text"]) for c in chunks]
    return keys, 1 - len(set(keys)) / len(keys)

keys, rate = dup_rate("Tell me the Pokedex entry for Gengar")
print(f"duplicate rate: {rate:.0%}")
for k in keys: print(" ", k[0])

In [ ]:
# THE FIX: delete duplicate points with the same doc_id + chunk_index, and keep one copy.
# The small viz collection gets the same dedup so the Web UI point cloud thins out too.
# (Reversible: `uv run python snapshot.py restore` brings the broken baseline back.)
from helpers.dedup import find_duplicate_ids

for col in (COLLECTION, config.VIZ_COLLECTION):
    before = client.count(col).count
    dup_ids = find_duplicate_ids(client, col)
    client.delete(collection_name=col, points_selector=dup_ids)
    print(f"{col}: {before} -> {client.count(col).count} points ({len(dup_ids)} duplicates deleted)")

In [ ]:
keys, rate = dup_rate("Tell me the Pokedex entry for Gengar")
print(f"duplicate rate: {rate:.0%}")   # now distinct documents fill the top-k
for k in keys: print(" ", k[0])

**Qdrant Web UI:** `pokemon_viz` in Visualize showed the duplicates as tight clusters; the dedup cleaned that collection too, so refresh and they thin out.

Dedup removes exact copies. Near-duplicate content can still survive, such as multiple generations of flavor text for the same species. That becomes a query-time diversity problem: use `group_by` or MMR when you need one result per document group.

## 3 · Fix #2: Upgrade The Embedding Model

**Dashboard read:** duplicates are gone, but clone-crowded paraphrase queries still miss. The 384d model returns lookalikes such as Togedemaru for Pikachu-style wording.

The model did not get worse. The corpus grew until the old vector space stopped separating close neighbors.

Qdrant named vectors make the migration safe: add the candidate model to the live collection, A/B it on the golden set, then commit or roll back with one retrieval-mode flip.

In [ ]:
# Pre-run offline. create_vector_name is instant; backfilling 23k points is the slow part
# and runs in the background. See prep.py. This is the zero-downtime migration:
#   client.create_vector_name(COLLECTION, config.DENSE_STRONG,
#       vector_name_config=models.DenseVectorNameConfig(
#           dense=models.DenseVectorConfig(size=1024, distance=models.Distance.COSINE)))
#   client.update_vectors(COLLECTION, points=[...])   # backfill, payload untouched
# Flip the agent to the candidate model in one line:
agent.set_retrieval(mode="bge")

rows = [json.loads(l) for l in Path("data/golden_dataset.jsonl").read_text().splitlines() if l.strip()]
fix2 = [r for r in rows if r["exercises"] == "fix2_embedding"]

# Instant gold-label check: recall@5 over unique docs (fragments of one doc fill several
# slots), same scorer as verify_arc.py. Judged scores live on the Future AGI dashboard.
def recall(mode):
    hit = 0
    for r in fix2:
        docs = []
        for c in agent.retrieve(r["query"], mode=mode, limit=30):
            if c["doc_id"] not in docs: docs.append(c["doc_id"])
        hit += int(bool(set(docs[:5]) & set(r["gold_doc_ids"])))
    return hit / len(fix2)

print(f"recall@5  small (MiniLM 384d)   = {recall('minilm'):.2f}")
print(f"recall@5  large (bge-large 1024d) = {recall('bge'):.2f}")

In [ ]:
# The eval confirms the win (0.64 -> 1.00 on the clone-crowded queries): keep bge.
# A regression would roll back with the same one-line flip — that's what the A/B protects.
print("retrieval mode:", agent.retrieval_state()["mode"])

The measurement is the lesson: the model that worked at 151 species stopped working at 1,025. The named-vector migration keeps both models live, so the decision is reversible and based on the repo's own golden set.

## 4 · Fix #3: Hybrid Retrieval + Late-Interaction Reranking

**Dashboard read:** the strong dense model still misses the hardest paraphrases. Groundedness may stay green if the agent refuses correctly, so recall against gold labels exposes the gap.

The fix is one `query_points` call: dense + sparse prefetch, RRF fusion, then ColBERT rerank. This builds on the bge migration; it does not replace it.

In [ ]:
q = "A Pokemon that lives in dark caves and uses sound waves to navigate and hunt."

# Dense-only — even the strong model — misses it: nothing in the query names Zubat,
# and the cave/echolocation wording is spread across many bat- and cave-dweller entries.
dense_only = [c["doc_id"] for c in agent.retrieve(q, mode="bge")]
print("dense-only   :", dense_only)

# The hybrid + rerank call. This is what mode='hybrid' runs. See agent.py.
sp = sparse([q], is_query=True)[0]
hybrid = client.query_points(
    COLLECTION,
    prefetch=[models.Prefetch(
        prefetch=[
            models.Prefetch(query=dense([q], config.MODEL_DENSE_STRONG, is_query=True)[0],
                            using=config.DENSE_STRONG, limit=20, params=agent.EXACT),
            models.Prefetch(query=models.SparseVector(indices=sp[0], values=sp[1]),
                            using=config.SPARSE, limit=20),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF), limit=20,
    )],
    query=colbert([q], is_query=True)[0], using=config.COLBERT, limit=5,
    with_payload=["doc_id"],
).points
print("hybrid+rerank:", [h.payload["doc_id"] for h in hybrid])

In [ ]:
# Flip the agent to hybrid, then score the queries dense retrieval was missing.
agent.set_retrieval(mode="hybrid")

import math
fix3 = [r for r in rows if r["exercises"] == "fix3_hybrid"]

# Instant gold-label check, same scorer as verify_arc.py. Judged scores live on the
# Future AGI dashboard.
def score(mode):
    rec, ndcg, mrr = 0, 0, 0
    for r in fix3:
        seen, ranked = set(), []
        for c in agent.retrieve(r["query"], mode=mode, limit=30):
            if c["doc_id"] not in seen: seen.add(c["doc_id"]); ranked.append(c["doc_id"])
        ranked, gold = ranked[:5], set(r["gold_doc_ids"])
        rec += int(any(d in gold for d in ranked))
        ndcg += sum(1/math.log2(i+2) for i,d in enumerate(ranked) if d in gold)  # idcg=1
        mrr += next((1/(i+1) for i,d in enumerate(ranked) if d in gold), 0)
    n = len(fix3); return rec/n, ndcg/n, mrr/n

print("dense (bge)   recall / NDCG@5 / MRR: %.2f / %.2f / %.2f" % score("bge"))
print("hybrid+rerank recall / NDCG@5 / MRR: %.2f / %.2f / %.2f" % score("hybrid"))

Measured stage by stage: at rank 20 the candidate pool already holds the right doc for 0.94 of queries, but RRF fusion alone scores WORSE at top-5 (0.72) than pure dense (0.78). The ColBERT rerank is what converts the pool into wins: recall@5 0.78 → 0.89, NDCG@5 0.61 → 0.77. Prefetch and rerank are a pipeline: prefetch builds the pool, the reranker orders it. Sparse earns its place on entity and keyword queries; this scored set is deliberately all paraphrases.

## 5 · Close The Cold Open With Metadata

Dedup, bge, and hybrid still do not fix the opening failure. The stale pre-Gen-6 chart matches the question too well, so it keeps winning retrieval.

Better retrieval cannot fix wrong-but-relevant data. The points already carry `is_current`; index it and filter on it.

In [ ]:
q = "Does the Steel type resist Ghost and Dark attacks?"
before = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("no filter :", before)

client.create_payload_index(COLLECTION, "is_current", models.PayloadSchemaType.BOOL)
agent.set_retrieval(current_only=True)
after = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("is_current:", after)

In [ ]:
# Ask the agent the cold-open question again. Now it is grounded in the current document.
answer, _ = agent.ask("Does the Steel type resist Ghost and Dark attacks?")
print(answer)

## 6 · Bonus: Multi-Hop In The Trace View

A compound question makes the agent retrieve twice. In Future AGI, this should show up as two retriever spans from one user request.

In [ ]:
answer, chunks = agent.ask("What does Drowzee eat, and is that Pokemon weak to Bug-type attacks?")
print(answer)